سلول ۱ — مسیرها و ورودی‌ها

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from collections import defaultdict

PROJECT_ROOT = Path(".")

TISCH_DIR = PROJECT_ROOT / "Data_raw" / "tisch2"
INTERIM_SCRNA_DIR = PROJECT_ROOT / "Data_interim" / "scrna"
NEG_DIR = PROJECT_ROOT / "Data_proc" / "negatives"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

for d in [INTERIM_SCRNA_DIR, NEG_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

E3_NEG_LOC_PATH = NEG_DIR / "negative_e3_after_ppi_loc_filter.csv"
DUB_NEG_LOC_PATH = NEG_DIR / "negative_dub_after_ppi_loc_filter.csv"
ALL_NEG_LOC_PATH = NEG_DIR / "negative_all_after_ppi_loc_filter.csv"

print("TISCH dir exists:", TISCH_DIR.exists(), TISCH_DIR)
print("E3 loc-filtered neg exists:", E3_NEG_LOC_PATH.exists())
print("DUB loc-filtered neg exists:", DUB_NEG_LOC_PATH.exists())
print("ALL loc-filtered neg exists:", ALL_NEG_LOC_PATH.exists())

سلول ۲ — پیدا کردن فایل‌های TISCH2

In [ ]:
all_tisch_files = sorted(TISCH_DIR.glob("**/*expression_Celltype_majorlineage.txt"))

crc_files = [p for p in all_tisch_files if p.name.upper().startswith("CRC_") or "/CRC/" in str(p)]
lihc_files = [p for p in all_tisch_files if p.name.upper().startswith("LIHC_") or "/LIHC/" in str(p)]

print("All TISCH expression files:", len(all_tisch_files))
print("CRC files:", len(crc_files))
for p in crc_files:
    print("  ", p.relative_to(PROJECT_ROOT))

print("\nLIHC files:", len(lihc_files))
for p in lihc_files:
    print("  ", p.relative_to(PROJECT_ROOT))

سلول ۳ — loader انعطاف‌پذیر TISCH2

In [ ]:
def read_any_table(path: Path) -> pd.DataFrame:
    for sep in [None, "\t", ",", ";"]:
        try:
            df = pd.read_csv(path, sep=sep, engine="python", dtype=str, low_memory=False)
            if df.shape[1] >= 2:
                return df
        except Exception:
            pass
    raise ValueError(f"Could not read table: {path}")


_gene_pat = re.compile(r"^[A-Za-z0-9][A-Za-z0-9\-\._]{1,}$")

def looks_like_gene_series(s: pd.Series) -> bool:
    s = s.dropna().astype(str).str.strip()
    if len(s) == 0:
        return False
    ok = s.str.match(_gene_pat) & (~s.str.contains(r"\s"))
    return ok.mean() >= 0.7


def normalize_gene_symbol(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    return x.upper()


def load_tisch_celltype_avg(path: Path) -> pd.DataFrame:
    """
    Output:
    cell_type × genes matrix, numeric.
    Rows = cell types.
    Columns = uppercase gene symbols.
    """
    df = read_any_table(path)
    
    if df.columns.astype(str).str.contains("Unnamed").all():
        df = pd.read_csv(path, sep=None, engine="python", header=None, low_memory=False)
    
    cols = list(df.columns)
    cols_l = [str(c).strip().lower() for c in cols]
    
    # Case A: explicit cell type column
    for alias in [
        "cell_type",
        "celltype",
        "cluster",
        "celltype_majorlineage",
        "celltype (major-lineage)",
        "celltype_major-lineage",
    ]:
        if alias in cols_l:
            ct_col = cols[cols_l.index(alias)]
            df = df.rename(columns={ct_col: "cell_type"})
            gene_cols = [c for c in df.columns if c != "cell_type"]
            rename_map = {c: normalize_gene_symbol(c) for c in gene_cols}
            df = df.rename(columns=rename_map)
            for c in df.columns:
                if c != "cell_type":
                    df[c] = pd.to_numeric(df[c], errors="coerce")
            out = df.set_index("cell_type").fillna(0.0)
            out = out.loc[:, ~out.columns.isna()]
            out = out.groupby(level=0).mean()
            return out
    
    # Case B: explicit gene column, transpose
    for alias in ["gene", "symbol", "genes", "gene_name", "rownames"]:
        if alias in cols_l:
            gcol = cols[cols_l.index(alias)]
            df = df.rename(columns={gcol: "GENE"})
            df["GENE"] = df["GENE"].map(normalize_gene_symbol)
            df = df.dropna(subset=["GENE"]).drop_duplicates(subset=["GENE"]).set_index("GENE")
            for c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
            out = df.T.fillna(0.0)
            out.index.name = "cell_type"
            return out
    
    # Case C: first column looks like genes, transpose
    first_col = df.columns[0]
    if looks_like_gene_series(df[first_col]):
        df = df.rename(columns={first_col: "GENE"})
        df["GENE"] = df["GENE"].map(normalize_gene_symbol)
        df = df.dropna(subset=["GENE"]).drop_duplicates(subset=["GENE"]).set_index("GENE")
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        out = df.T.fillna(0.0)
        out.index.name = "cell_type"
        return out
    
    raise ValueError(
        f"Could not infer TISCH2 format for {path.name}. "
        f"Columns: {list(df.columns)[:20]}"
    )

سلول ۴ — بارگذاری ماتریس‌های CRC و LIHC

In [ ]:
def load_many_tisch(files, cancer_name):
    mats = []
    qc_rows = []
    
    for path in files:
        try:
            M = load_tisch_celltype_avg(path)
            M = M.loc[:, ~M.columns.duplicated()].copy()
            M.index = [f"{cancer_name}|{path.stem}|{idx}" for idx in M.index.astype(str)]
            
            mats.append(M)
            
            qc_rows.append({
                "cancer": cancer_name,
                "file": str(path.relative_to(PROJECT_ROOT)),
                "loaded": True,
                "n_celltypes": M.shape[0],
                "n_genes": M.shape[1],
                "error": "",
            })
            
            print(f"Loaded {cancer_name}: {path.name} -> {M.shape}")
        
        except Exception as e:
            qc_rows.append({
                "cancer": cancer_name,
                "file": str(path.relative_to(PROJECT_ROOT)),
                "loaded": False,
                "n_celltypes": 0,
                "n_genes": 0,
                "error": str(e),
            })
            
            print(f"FAILED {cancer_name}: {path.name} -> {e}")
    
    return mats, pd.DataFrame(qc_rows)


crc_mats, crc_qc = load_many_tisch(crc_files, "CRC")
lihc_mats, lihc_qc = load_many_tisch(lihc_files, "LIHC")

tisch_load_qc = pd.concat([crc_qc, lihc_qc], ignore_index=True)
display(tisch_load_qc)

tisch_load_qc.to_csv(
    QC_DIR / "tisch2_load_qc.csv",
    index=False
)

print("CRC matrices loaded:", len(crc_mats))
print("LIHC matrices loaded:", len(lihc_mats))

داریم دنبال اینکه چرا لود نشدن میگردیم

In [ ]:
display(tisch_load_qc)

print("Failed files:")
display(tisch_load_qc[tisch_load_qc["loaded"] == False][["cancer", "file", "error"]])

In [ ]:
from pathlib import Path
import pandas as pd

sample_files = crc_files[:2] + lihc_files[:2]

for path in sample_files:
    print("\n" + "="*120)
    print("FILE:", path)
    print("NAME:", path.name)
    
    print("\n--- raw first 10 lines ---")
    with open(path, "r", errors="replace") as f:
        for i in range(10):
            line = f.readline()
            if not line:
                break
            print(f"{i+1}: {line[:500].rstrip()}")
    
    for sep_name, sep in [("tab", "\t"), ("comma", ","), ("auto", None)]:
        print(f"\n--- read sep={sep_name} ---")
        try:
            df = pd.read_csv(path, sep=sep, engine="python", dtype=str, nrows=5)
            print("shape:", df.shape)
            print("columns:", df.columns.tolist()[:20])
            display(df.head())
        except Exception as e:
            print("FAILED:", e)

In [ ]:
all_files = sorted([p for p in TISCH_DIR.glob("**/*") if p.is_file()])

print("Total files:", len(all_files))
for p in all_files[:100]:
    print(p.relative_to(PROJECT_ROOT))

سلول ۳ جدید — loader مخصوص فایل‌های TISCH2 تو

In [ ]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

_gene_pat = re.compile(r"^[A-Za-z0-9][A-Za-z0-9\-\._]{1,}$")


def normalize_gene_symbol(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    return x.upper()


def looks_like_gene_index(index) -> bool:
    vals = pd.Series(index).dropna().astype(str).str.strip()
    if len(vals) == 0:
        return False
    
    # gene-like: no spaces, mostly alphanumeric with -, ., _
    ok = vals.str.match(_gene_pat) & (~vals.str.contains(r"\s"))
    return ok.mean() >= 0.6


def load_tisch_celltype_avg(path: Path) -> pd.DataFrame:
    """
    Load TISCH2 expression_Celltype_majorlineage.txt files.

    Expected common format in your files:
        rows    = genes
        columns = cell types

    Output:
        rows    = cell types
        columns = uppercase gene symbols
        values  = numeric expression
    """
    
    # First try: normal tab read
    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        low_memory=False,
    )
    
    # Important case:
    # Pandas may automatically put gene names into index when header has one fewer field.
    if looks_like_gene_index(df.index):
        mat = df.copy()
        mat.index = [normalize_gene_symbol(x) for x in mat.index]
        mat = mat[~pd.isna(mat.index)]
        mat = mat[~mat.index.duplicated(keep="first")]
        
        for c in mat.columns:
            mat[c] = pd.to_numeric(mat[c], errors="coerce")
        
        # transpose: cell types × genes
        out = mat.T.fillna(0.0)
        out.index.name = "cell_type"
        return out
    
    # Case: first column is gene column
    first_col = df.columns[0]
    first_vals = df[first_col].dropna().astype(str).head(50)
    
    if looks_like_gene_index(first_vals):
        mat = df.copy()
        mat = mat.rename(columns={first_col: "GENE"})
        mat["GENE"] = mat["GENE"].map(normalize_gene_symbol)
        mat = mat.dropna(subset=["GENE"])
        mat = mat.drop_duplicates(subset=["GENE"])
        mat = mat.set_index("GENE")
        
        for c in mat.columns:
            mat[c] = pd.to_numeric(mat[c], errors="coerce")
        
        out = mat.T.fillna(0.0)
        out.index.name = "cell_type"
        return out
    
    # Case: explicit gene column
    cols_lower = [str(c).strip().lower() for c in df.columns]
    gene_aliases = ["gene", "genes", "symbol", "gene_symbol", "gene_name", "rownames"]
    
    for alias in gene_aliases:
        if alias in cols_lower:
            gcol = df.columns[cols_lower.index(alias)]
            mat = df.rename(columns={gcol: "GENE"})
            mat["GENE"] = mat["GENE"].map(normalize_gene_symbol)
            mat = mat.dropna(subset=["GENE"])
            mat = mat.drop_duplicates(subset=["GENE"])
            mat = mat.set_index("GENE")
            
            for c in mat.columns:
                mat[c] = pd.to_numeric(mat[c], errors="coerce")
            
            out = mat.T.fillna(0.0)
            out.index.name = "cell_type"
            return out
    
    raise ValueError(
        f"Could not infer TISCH2 format for {path.name}. "
        f"shape={df.shape}, columns={list(df.columns)[:20]}, "
        f"index_sample={list(df.index[:5])}"
    )

سلول ۴ جدید — انتخاب فایل‌های درست و حذف تکراری‌ها

In [ ]:
all_tisch_files = sorted([
    p for p in TISCH_DIR.glob("**/*expression_Celltype_majorlineage.txt")
    if p.is_file() and not p.name.startswith(".")
])

crc_files = []
lihc_files = []

for p in all_tisch_files:
    s = str(p).upper()
    name = p.name.upper()
    
    if "CRC_" in name or "/CRC/" in s:
        crc_files.append(p)
    elif "LIHC_" in name or "/LIHC/" in s:
        lihc_files.append(p)

# Remove exact duplicate paths
crc_files = sorted(set(crc_files))
lihc_files = sorted(set(lihc_files))

print("CRC files:", len(crc_files))
for p in crc_files:
    print("  ", p.relative_to(PROJECT_ROOT))

print("\nLIHC files:", len(lihc_files))
for p in lihc_files:
    print("  ", p.relative_to(PROJECT_ROOT))

سلول ۵  — بارگذاری ماتریس‌ها

In [ ]:
def load_many_tisch(files, cancer_name):
    mats = []
    qc_rows = []
    
    for path in files:
        try:
            M = load_tisch_celltype_avg(path)
            M = M.loc[:, ~M.columns.duplicated()].copy()
            
            # make celltype index unique and traceable
            M.index = [
                f"{cancer_name}|{path.stem}|{idx}"
                for idx in M.index.astype(str)
            ]
            
            mats.append(M)
            
            qc_rows.append({
                "cancer": cancer_name,
                "file": str(path.relative_to(PROJECT_ROOT)),
                "loaded": True,
                "n_celltypes": M.shape[0],
                "n_genes": M.shape[1],
                "error": "",
            })
            
            print(f"Loaded {cancer_name}: {path.name} -> {M.shape}")
        
        except Exception as e:
            qc_rows.append({
                "cancer": cancer_name,
                "file": str(path.relative_to(PROJECT_ROOT)),
                "loaded": False,
                "n_celltypes": 0,
                "n_genes": 0,
                "error": str(e),
            })
            
            print(f"FAILED {cancer_name}: {path.name} -> {e}")
    
    return mats, pd.DataFrame(qc_rows)


crc_mats, crc_qc = load_many_tisch(crc_files, "CRC")
lihc_mats, lihc_qc = load_many_tisch(lihc_files, "LIHC")

tisch_load_qc = pd.concat([crc_qc, lihc_qc], ignore_index=True)

display(tisch_load_qc)

tisch_load_qc.to_csv(
    QC_DIR / "tisch2_load_qc.csv",
    index=False
)

print("CRC matrices loaded:", len(crc_mats))
print("LIHC matrices loaded:", len(lihc_mats))
print("Failed:", (tisch_load_qc["loaded"] == False).sum())

سلول ۶ — ادغام ماتریس‌های CRC و LIHC

In [ ]:
def combine_mats(mats):
    if not mats:
        return pd.DataFrame()
    
    all_genes = sorted(set().union(*[set(M.columns) for M in mats]))
    
    aligned = []
    for M in mats:
        aligned.append(M.reindex(columns=all_genes, fill_value=0.0))
    
    out = pd.concat(aligned, axis=0)
    out = out.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return out


crc_expr = combine_mats(crc_mats)
lihc_expr = combine_mats(lihc_mats)

print("CRC combined:", crc_expr.shape)
print("LIHC combined:", lihc_expr.shape)

crc_expr.to_csv(
    INTERIM_SCRNA_DIR / "tisch2_crc_expression_by_celltype.csv"
)

lihc_expr.to_csv(
    INTERIM_SCRNA_DIR / "tisch2_lihc_expression_by_celltype.csv"
)

display(crc_expr.head())
display(lihc_expr.head())

سلول ۷ — خواندن نگاتیوهای بعد از PPI و localization

In [ ]:
e3_neg_loc = pd.read_csv(E3_NEG_LOC_PATH, dtype=str, low_memory=False)
dub_neg_loc = pd.read_csv(DUB_NEG_LOC_PATH, dtype=str, low_memory=False)
negative_all_loc = pd.read_csv(ALL_NEG_LOC_PATH, dtype=str, low_memory=False)

for df in [e3_neg_loc, dub_neg_loc, negative_all_loc]:
    df["label"] = df["label"].astype(int)

print("E3 after PPI+loc:", e3_neg_loc.shape)
print("DUB after PPI+loc:", dub_neg_loc.shape)
print("ALL after PPI+loc:", negative_all_loc.shape)

display(e3_neg_loc.head())
display(dub_neg_loc.head())

سلول ۸ — محاسبه co-expression score

In [ ]:
Q_THR = 0.70
ABS_THR = 1.0

def get_gene_vector(expr: pd.DataFrame, gene: str):
    gene = normalize_gene_symbol(gene)
    if pd.isna(gene) or expr.empty or gene not in expr.columns:
        return None
    return expr[gene].astype(float)


def coexpression_in_expr(expr: pd.DataFrame, geneA: str, geneB: str, q_thr=0.70, abs_thr=1.0):
    """
    Returns dict with:
    has_both_genes, coexpr_flag, max_min_expr, best_celltype
    """
    vA = get_gene_vector(expr, geneA)
    vB = get_gene_vector(expr, geneB)
    
    if vA is None or vB is None:
        return {
            "has_both_genes": False,
            "coexpr_flag": False,
            "max_min_expr": 0.0,
            "best_celltype": "NA",
        }
    
    thrA = max(float(vA.quantile(q_thr)), abs_thr)
    thrB = max(float(vB.quantile(q_thr)), abs_thr)
    
    min_expr = np.minimum(vA.values, vB.values)
    max_min_expr = float(np.max(min_expr)) if len(min_expr) else 0.0
    
    both_high = (vA >= thrA) & (vB >= thrB)
    
    if bool(both_high.any()):
        best_idx = np.argmax(min_expr)
        best_celltype = str(expr.index[best_idx])
        return {
            "has_both_genes": True,
            "coexpr_flag": True,
            "max_min_expr": max_min_expr,
            "best_celltype": best_celltype,
        }
    
    best_idx = np.argmax(min_expr) if len(min_expr) else None
    best_celltype = str(expr.index[best_idx]) if best_idx is not None else "NA"
    
    return {
        "has_both_genes": True,
        "coexpr_flag": False,
        "max_min_expr": max_min_expr,
        "best_celltype": best_celltype,
    }


def score_pair_scrna(geneA: str, geneB: str):
    crc_res = coexpression_in_expr(crc_expr, geneA, geneB, Q_THR, ABS_THR)
    lihc_res = coexpression_in_expr(lihc_expr, geneA, geneB, Q_THR, ABS_THR)
    
    coexpr_crc = crc_res["coexpr_flag"]
    coexpr_lihc = lihc_res["coexpr_flag"]
    
    if crc_res["max_min_expr"] >= lihc_res["max_min_expr"]:
        best_cancer = "CRC"
        best_celltype = crc_res["best_celltype"]
        best_score = crc_res["max_min_expr"]
    else:
        best_cancer = "LIHC"
        best_celltype = lihc_res["best_celltype"]
        best_score = lihc_res["max_min_expr"]
    
    return {
        "scrna_has_both_genes_crc": crc_res["has_both_genes"],
        "scrna_has_both_genes_lihc": lihc_res["has_both_genes"],
        "scrna_coexpr_crc": coexpr_crc,
        "scrna_coexpr_lihc": coexpr_lihc,
        "scrna_coexpr_flag": bool(coexpr_crc or coexpr_lihc),
        "scrna_max_min_expr": best_score,
        "scrna_best_cancer": best_cancer,
        "scrna_best_celltype": best_celltype,
    }

سلول ۹ — اعمال فیلتر scRNA

In [ ]:
def annotate_scrna(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    rows = []
    
    for i, r in df.iterrows():
        if i % 500 == 0:
            print(f"{dataset_name}: {i}/{len(df)}")
        
        geneA = r.get("enz_gene", np.nan)
        geneB = r.get("sub_gene", np.nan)
        
        res = score_pair_scrna(geneA, geneB)
        rows.append(res)
    
    ann = pd.DataFrame(rows)
    out = pd.concat([df.reset_index(drop=True), ann], axis=1)
    return out


e3_scrna_annot = annotate_scrna(e3_neg_loc, "E3")
dub_scrna_annot = annotate_scrna(dub_neg_loc, "DUB")

e3_neg_after_scrna = e3_scrna_annot[~e3_scrna_annot["scrna_coexpr_flag"]].copy()
dub_neg_after_scrna = dub_scrna_annot[~dub_scrna_annot["scrna_coexpr_flag"]].copy()

e3_removed_scrna = e3_scrna_annot[e3_scrna_annot["scrna_coexpr_flag"]].copy()
dub_removed_scrna = dub_scrna_annot[dub_scrna_annot["scrna_coexpr_flag"]].copy()

negative_all_after_scrna = pd.concat(
    [e3_neg_after_scrna, dub_neg_after_scrna],
    ignore_index=True,
)

removed_all_scrna = pd.concat(
    [e3_removed_scrna, dub_removed_scrna],
    ignore_index=True,
)

print("E3 before:", len(e3_neg_loc), "removed:", len(e3_removed_scrna), "after:", len(e3_neg_after_scrna))
print("DUB before:", len(dub_neg_loc), "removed:", len(dub_removed_scrna), "after:", len(dub_neg_after_scrna))
print("ALL before:", len(negative_all_loc), "removed:", len(removed_all_scrna), "after:", len(negative_all_after_scrna))

display(removed_all_scrna.head())

سلول ۱۰ — QC فیلتر scRNA

In [ ]:
def qc_after_scrna(before_df, after_df, removed_df, name):
    return {
        "dataset": name,
        "n_before": len(before_df),
        "n_removed_by_scrna": len(removed_df),
        "n_after": len(after_df),
        "removed_fraction": len(removed_df) / len(before_df) if len(before_df) else np.nan,
        "n_after_unique_pair_id": after_df["pair_id"].nunique(),
        "n_after_duplicate_pair_id_rows": int(after_df.duplicated("pair_id").sum()),
        "n_after_missing_enzyme_class": int(after_df["enzyme_class"].isna().sum()),
        "n_after_missing_enz_ac": int(after_df["enz_ac"].isna().sum()),
        "n_after_missing_sub_ac": int(after_df["sub_ac"].isna().sum()),
        "n_after_pair_id_starts_with_nan": int(after_df["pair_id"].astype(str).str.startswith("nan|").sum()),
        "n_after_label_0": int((after_df["label"].astype(int) == 0).sum()),
        "n_removed_crc": int(removed_df["scrna_coexpr_crc"].sum()) if len(removed_df) else 0,
        "n_removed_lihc": int(removed_df["scrna_coexpr_lihc"].sum()) if len(removed_df) else 0,
        "n_after_has_both_genes_crc": int(after_df["scrna_has_both_genes_crc"].sum()) if len(after_df) else 0,
        "n_after_has_both_genes_lihc": int(after_df["scrna_has_both_genes_lihc"].sum()) if len(after_df) else 0,
    }


scrna_filter_qc = pd.DataFrame([
    qc_after_scrna(e3_neg_loc, e3_neg_after_scrna, e3_removed_scrna, "E3_negative_after_ppi_loc_scrna"),
    qc_after_scrna(dub_neg_loc, dub_neg_after_scrna, dub_removed_scrna, "DUB_negative_after_ppi_loc_scrna"),
    qc_after_scrna(negative_all_loc, negative_all_after_scrna, removed_all_scrna, "ALL_negative_after_ppi_loc_scrna"),
])

display(scrna_filter_qc)

print("Label counts after scRNA:")
print(negative_all_after_scrna["label"].value_counts(dropna=False))

print("Duplicate pair_id after scRNA:", negative_all_after_scrna.duplicated("pair_id").sum())

print("\nRemoved by cancer flags:")
if len(removed_all_scrna):
    print(removed_all_scrna[["scrna_coexpr_crc", "scrna_coexpr_lihc"]].value_counts())
else:
    print("No rows removed by scRNA.")

 ذخیره کن

In [ ]:
# Save scRNA-filtered negatives
e3_neg_after_scrna.to_csv(
    NEG_DIR / "negative_e3_after_ppi_loc_scrna_filter.csv",
    index=False
)

dub_neg_after_scrna.to_csv(
    NEG_DIR / "negative_dub_after_ppi_loc_scrna_filter.csv",
    index=False
)

negative_all_after_scrna.to_csv(
    NEG_DIR / "negative_all_after_ppi_loc_scrna_filter.csv",
    index=False
)

# Save removed by scRNA
e3_removed_scrna.to_csv(
    QC_DIR / "negative_e3_removed_by_scrna.csv",
    index=False
)

dub_removed_scrna.to_csv(
    QC_DIR / "negative_dub_removed_by_scrna.csv",
    index=False
)

removed_all_scrna.to_csv(
    QC_DIR / "negative_all_removed_by_scrna.csv",
    index=False
)

# Save annotated versions before filtering
e3_scrna_annot.to_csv(
    NEG_DIR / "negative_e3_after_ppi_loc_scrna_annot.csv",
    index=False
)

dub_scrna_annot.to_csv(
    NEG_DIR / "negative_dub_after_ppi_loc_scrna_annot.csv",
    index=False
)

negative_all_scrna_annot = pd.concat(
    [e3_scrna_annot, dub_scrna_annot],
    ignore_index=True
)

negative_all_scrna_annot.to_csv(
    NEG_DIR / "negative_all_after_ppi_loc_scrna_annot.csv",
    index=False
)

# Save QC
scrna_filter_qc.to_csv(
    QC_DIR / "scrna_filter_qc.csv",
    index=False
)

print("Saved:")
print(NEG_DIR / "negative_e3_after_ppi_loc_scrna_filter.csv")
print(NEG_DIR / "negative_dub_after_ppi_loc_scrna_filter.csv")
print(NEG_DIR / "negative_all_after_ppi_loc_scrna_filter.csv")
print(NEG_DIR / "negative_all_after_ppi_loc_scrna_annot.csv")
print(QC_DIR / "negative_all_removed_by_scrna.csv")
print(QC_DIR / "scrna_filter_qc.csv")

In [ ]:
check_files = [
    NEG_DIR / "negative_e3_after_ppi_loc_scrna_filter.csv",
    NEG_DIR / "negative_dub_after_ppi_loc_scrna_filter.csv",
    NEG_DIR / "negative_all_after_ppi_loc_scrna_filter.csv",
    NEG_DIR / "negative_e3_after_ppi_loc_scrna_annot.csv",
    NEG_DIR / "negative_dub_after_ppi_loc_scrna_annot.csv",
    NEG_DIR / "negative_all_after_ppi_loc_scrna_annot.csv",
    QC_DIR / "negative_e3_removed_by_scrna.csv",
    QC_DIR / "negative_dub_removed_by_scrna.csv",
    QC_DIR / "negative_all_removed_by_scrna.csv",
    QC_DIR / "scrna_filter_qc.csv",
    QC_DIR / "tisch2_load_qc.csv",
    INTERIM_SCRNA_DIR / "tisch2_crc_expression_by_celltype.csv",
    INTERIM_SCRNA_DIR / "tisch2_lihc_expression_by_celltype.csv",
]

for f in check_files:
    print(f.name, "exists:", f.exists())
    if f.exists() and f.suffix == ".csv":
        tmp = pd.read_csv(f, dtype=str, low_memory=False)
        print("shape:", tmp.shape)